# 🧠 VisionOps — Train → Track → Export → Predict

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<your-username>/visionops/blob/main/notebooks/colab_demo.ipynb)

This notebook is a **thin demo** on top of the `visionops` package (the FSDL lecture's *"write modules, import them in notebooks"* pattern).
All the real logic lives in `src/visionops/`, so it is tested, linted and reusable.

**What you'll do**
1. Set up the repo on a free Colab GPU
2. Configure the experiment and look at the data
3. Train a pretrained ResNet-18 on CIFAR-10 (PyTorch Lightning + timm)
4. Plot learning curves
5. Check the exported **ONNX** model and run torch-free inference
6. *(Optional)* track runs with Weights & Biases

> ⚠️ Switch the runtime first: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import torch

print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Training on CPU works but is very slow.")

## 1. Get the code
**Option A (recommended):** clone from GitHub — set your username below.
**Option B:** upload `visionops.zip` (uncomment the second block).

In [ ]:
import os
import subprocess
import sys

GITHUB_USER = "<your-username>"  # <-- change this
REPO = f"https://github.com/{GITHUB_USER}/visionops.git"

if os.path.basename(os.getcwd()) != "visionops":
    if not os.path.exists("visionops"):
        subprocess.run(["git", "clone", "-q", REPO], check=True)
    os.chdir("visionops")

# --- Option B: upload the zip instead ---
# from google.colab import files
# import zipfile
# uploaded = files.upload()  # choose visionops.zip
# zipfile.ZipFile(next(iter(uploaded))).extractall(".")
# os.chdir("visionops")

print("Working directory:", os.getcwd())

In [ ]:
%pip install -q -e ".[train]"
sys.path.insert(0, os.path.abspath("src"))  # make the import work without restarting the kernel

## 2. Configure the experiment
Every field can be overridden here or from the CLI (`--lr 3e-4`).

In [ ]:
from visionops.config import load_config

cfg = load_config("configs/default.yaml", {"max_epochs": 5, "run_name": "colab-baseline"})
cfg

## 3. Look at the data

In [ ]:
import matplotlib.pyplot as plt
import torch

from visionops.data import CLASSES, MEAN, STD, CIFAR10DataModule

dm = CIFAR10DataModule(
    dataset=cfg.dataset, data_dir=cfg.data_dir, image_size=cfg.image_size,
    batch_size=16, num_workers=2, val_split=cfg.val_split, seed=cfg.seed,
)
dm.prepare_data()
dm.setup()
x, y = next(iter(dm.val_dataloader()))

mean = torch.tensor(MEAN).view(1, 3, 1, 1)
std = torch.tensor(STD).view(1, 3, 1, 1)
images = (x * std + mean).clamp(0, 1)  # undo normalisation for display

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, img, label in zip(axes.flat, images, y):
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(CLASSES[label], fontsize=9)
    ax.axis("off")
plt.suptitle(f"CIFAR-10 validation samples (resized to {cfg.image_size}x{cfg.image_size})")
plt.show()

## 4. Train
`run(cfg)` trains with early stopping, evaluates the **best** checkpoint on the test set, and exports `model.onnx` with a PyTorch-vs-ONNX parity check.

In [ ]:
from visionops.train import run

results = run(cfg)
print(f"\nTest accuracy: {results['test_acc']:.2%}")
print("Best checkpoint:", results["best_checkpoint"])
print("ONNX model:", results["onnx_path"])

## 5. Learning curves

In [ ]:
import pandas as pd

metrics = pd.read_csv(os.path.join(results["log_dir"], "metrics.csv"))
cols = [c for c in ["train_loss", "val_loss", "train_acc", "val_acc"] if c in metrics.columns]
per_epoch = metrics.groupby("epoch")[cols].first()  # CSVLogger writes sparse rows

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
per_epoch[[c for c in cols if "loss" in c]].plot(ax=axes[0], marker="o", title="Loss")
per_epoch[[c for c in cols if "acc" in c]].plot(ax=axes[1], marker="o", title="Accuracy")
plt.tight_layout()
plt.show()

## 6. Torch-free inference with ONNX Runtime
This is exactly what the Docker serving image runs — no PyTorch needed.

In [ ]:
import numpy as np
from torchvision import datasets

from visionops.inference import Predictor

predictor = Predictor(results["onnx_path"])
test_set = datasets.CIFAR10(cfg.data_dir, train=False)  # raw PIL images

# Accuracy of the ONNX model on the first 500 test images
hits = [predictor.predict(test_set[i][0], top_k=1)[0][0] == CLASSES[test_set[i][1]] for i in range(500)]
print(f"ONNX accuracy on first 500 test images: {np.mean(hits):.2%}")

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax, i in zip(axes.flat, range(12)):
    img, label = test_set[i]
    name, prob = predictor.predict(img, top_k=1)[0]
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"{name} ({prob:.0%})\ntrue: {CLASSES[label]}", fontsize=9,
                 color="green" if name == CLASSES[label] else "red")
plt.show()

## 7. (Optional) Track runs with Weights & Biases
Create a free account at [wandb.ai](https://wandb.ai), then uncomment. Every run gets logged with its full config, so you can compare experiments later.

In [ ]:
# %pip install -q wandb
# import wandb
# wandb.login()
# cfg_wandb = load_config("configs/default.yaml", {"logger": "wandb", "run_name": "colab-wandb", "max_epochs": 5})
# results_wandb = run(cfg_wandb)

## 8. Download the exported model

In [ ]:
try:
    from google.colab import files

    files.download(results["onnx_path"])
    files.download(results["onnx_path"].replace(".onnx", "_meta.json"))
except ImportError:
    print("Not running in Colab. Files are in:", os.path.dirname(results["onnx_path"]))

## Next steps
- Put `model.onnx` and `model_meta.json` into the repo's `models/` folder and run `make docker-build && make docker-run` to serve a Streamlit demo.
- Run a **hyper-parameter sweep**: `wandb sweep sweeps/sweep.yaml`.
- Try another backbone: `load_config(..., {"backbone": "convnext_tiny"})` (any [`timm`](https://github.com/huggingface/pytorch-image-models) model).
- Train on multiple GPUs: `--devices 4 --strategy ddp` (from the CLI, not from a notebook).